In [12]:
from langchain_ollama import OllamaLLM
from langchain_core.output_parsers import StrOutputParser, SimpleJsonOutputParser
from langchain_core.prompts import PromptTemplate

llm = OllamaLLM(
    model="qwen2.5-coder:7b",
    temperature=0
)

system_prompt = PromptTemplate.from_template(
    """ 
    Give sentiment and themes for the given input review.

    Give output as json with "sentiment" key and value should be either of [POSITIVE, NEUTRAL, NEGATIVE]. 
    And another key "themes" with list of themes talked about in the review, for example "Producut Issue", "Packaging Issue", etc.

    Strictly only give json output as string and nothing else. Do not give markdown.

    Input Review:
    ```
    {input_review}
    ```
    """
)


In [13]:
parser = SimpleJsonOutputParser()
chain = system_prompt | llm | parser

In [14]:
response = chain.invoke(
    input={
        "input_review": "This bedsheet is beautiful and looks just like in the picture. The print is very pretty. It is really comfortable and smooth. Fabric is very nice , color doesn’t fade even after the wash. It fits our queen size bed perfectly ( it can be easy used for King size bed) . Go for it, value for money."
    }
)

In [15]:
response

{'sentiment': 'POSITIVE',
 'themes': ['Product Quality',
  'Design',
  'Comfort',
  'Durability',
  'Size',
  'Value for Money']}

In [16]:
response["sentiment"]

'POSITIVE'

In [17]:
multiple_review_analysis = []

reviews = ["This bedsheet is beautiful and looks just like in the picture. The print is very pretty. It is really comfortable and smooth. Fabric is very nice , color doesn’t fade even after the wash. It fits our queen size bed perfectly ( it can be easy used for King size bed) . Go for it, value for money.",
           "Quality is good. But it is not a king size one. I'd say it is queen size.",
           "Superb bedsheet.. soft material.. I like it . Go for it .. colour fast febric nd it looked so beautiful 😻 finally got what I need . Will order more from this brand.. happy shopping 😀🛍️",
           "In the first wash, the bedsheet faded and shrunk a lot. Don't buy this product.",
           "Product is ok for discounted price. Colour is not that bright as shown in pic...some slight defects on bedsheet....ok ok product"
          ]

for every_review in reviews:
    response = chain.invoke(
        input={
            "input_review": every_review
        }
    )
    multiple_review_analysis.append(response)

print(multiple_review_analysis)

[{'sentiment': 'POSITIVE', 'themes': ['Product Quality', 'Design', 'Comfort', 'Durability', 'Size', 'Value for Money']}, {'sentiment': 'NEUTRAL', 'themes': ['Quality', 'Size']}, {'sentiment': 'POSITIVE', 'themes': ['Product Quality', 'Customer Satisfaction', 'Brand Loyalty']}, {'sentiment': 'NEGATIVE', 'themes': ['Product Issue', 'Fading', 'Shrinking']}, {'sentiment': 'NEUTRAL', 'themes': ['Product Issue', 'Price', 'Color', 'Defects']}]


In [18]:
actionable_insights_prompt = PromptTemplate.from_template("""I have sentiments around some themes extracted from reviews.
I want you to give me actionable insights in bullet points.
Keep it exceptionally very short and crisp.

Input:
```
{review_analysis}
```""")

In [19]:
str_parser = StrOutputParser()
insights_chain = actionable_insights_prompt | llm | str_parser

In [20]:
response = insights_chain.invoke(input={
    "review_analysis": multiple_review_analysis
})

In [21]:
print(response)

- **Positive Insights:**
  - **Product Quality:** High satisfaction with product quality.
  - **Design & Comfort:** Positive feedback on design and comfort.
  - **Durability:** Customers appreciate the product's durability.
  - **Customer Satisfaction & Brand Loyalty:** Strong customer satisfaction and brand loyalty.
  - **Value for Money:** Products are considered good value for money.

- **Neutral Insights:**
  - **Quality & Size:** Neutral sentiment on product quality and size.
  - **Price & Color:** Neutral feedback on price and color.
  - **Defects:** Neutral comments on product defects.

- **Negative Insights:**
  - **Product Issue:** Issues with product functionality.
  - **Fading & Shrinking:** Negative feedback on fading and shrinking.
